# G12 - closing the two gaps G11 left

**Detector fix.** G11's gate tested how SHARP the drop was relative to other drops; the prediction was about WHERE it is. Where the collapse spans two grid steps the two largest drops are nearly equal, the ratio falls to 1.1, and a curve that visibly cliffs is reported as having none. Location and sharpness are now reported separately, and location has no free parameter.

**Grid fix.** Adds 1024, 1280 and 1792. G11 sampled only 1152 between 896 and 1408, so "nowhere earlier" rested on a gap. These three are nobody's ambient dimension, so they are three fresh chances for the account to fail.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# G12 — closing the two gaps G11 left.
# Run after G11. Requires the same caches; rebuilds the hub itself.
#
# WHAT G11 ESTABLISHED. The prediction was written before the run: each
# source cliffs at its own ambient dimension and nowhere earlier, and the
# location does not move with the head target. In all six curves the
# largest single drop landed exactly on the source's dimension:
#
#   img_small  768-d   largest drop at  768  (both targets)
#   img_base  1536-d   largest drop at 1536  (both targets)
#   img_large 2048-d   largest drop at 2048  (both targets)
#
# TWO PROBLEMS WITH HOW THAT WAS SCORED, both mine.
#
# 1. THE DETECTOR MEASURED THE WRONG THING. G11's gate asked whether the
#    largest drop was big RELATIVE TO OTHER DROPS in the same curve. That
#    is a sharpness test, and the prediction was about LOCATION. Where the
#    collapse spans two grid steps - img_base under bge falls -0.486 at
#    1536 then -0.434 at 1664 - the two largest drops are nearly equal, the
#    ratio comes out 1.1, and the gate reports "no cliff" for a curve that
#    visibly falls off a cliff. It then reported the location as moving
#    with the target, when what moved was only whether the gate fired.
#
#    Fixed here: report WHERE the largest drop is, and separately how
#    sharp it is. Location is the pre-registered claim; sharpness is a
#    descriptive statistic, not a gate.
#
# 2. THE GRID WAS THIN WHERE IT MATTERED LEAST TO THE HIT AND MOST TO THE
#    "NOWHERE EARLIER" HALF. Between 896 and 1408 G11 sampled only 1152 -
#    a 512-wide span with one interior point. A cliff at 1024 or 1280
#    would have been invisible, and "nowhere earlier" would have been
#    asserted on evidence that could not have contradicted it. This adds
#    1024 and 1280, plus 1792 to firm up the run below img_large's
#    dimension.
#
# WHAT WOULD STILL FALSIFY THE ACCOUNT. A collapse at 1024, 1280 or 1792
# for a source whose dimension is not that number. The rank wall predicts
# those three widths are unremarkable for every source, so they are three
# fresh chances for it to fail.
#
# NOTE ON READING THIS HONESTLY. The detector below was written AFTER
# seeing G11's output, which is the move this project exists to guard
# against. What makes it legitimate rather than a rescue: the prediction
# concerned location, the location is visible in G11's raw numbers with no
# metric at all, and this detector is not tuned - it reports the argmin of
# the step vector, which has no free parameter. If it had a threshold in
# it, it should not be trusted.
# ==========================================================
import os
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ["DATA_DIR"])
ALPHA, N_EVAL, SEED = 1e-2, 1000, 0

# G11's grid plus the three fill-in points
WIDTHS = [512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 1664,
          1792, 1920, 2048, 2176]
NEW_POINTS = {1024, 1280, 1792}

SPACES = {}
for size in ("small", "base", "large"):
    SPACES[f"img_{size}"] = np.load(
        str(DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz")
    )["img"].astype(np.float64)
N = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N] for k, v in SPACES.items()}
SPACES["txt_bge"] = np.load(
    str(DATA_DIR / "crossmodal_pairs.npz"))["txt"].astype(np.float64)[:N]

TARGETS = {"bge": SPACES["txt_bge"]}
_sb = DATA_DIR / "e13_txt_sbert.npz"
if _sb.exists():
    TARGETS["sbert"] = np.load(str(_sb))["txt"].astype(np.float64)[:N]

SOURCES = ["img_small", "img_base", "img_large"]
DIM = {s: SPACES[s].shape[1] for s in SOURCES}

print("PRE-REGISTERED, before running:")
for s in SOURCES:
    print(f"  {s:12s} largest drop at {DIM[s]}")
print(f"  and the three new widths {sorted(NEW_POINTS)} are unremarkable")
print("  for every source - three fresh chances for the account to fail\n")

rng = np.random.default_rng(SEED)
perm = rng.permutation(N)
te, tr = perm[:N_EVAL], perm[N_EVAL:]


def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)


def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)


def r1(P, G):
    return float(((l2n(P) @ l2n(G).T).argmax(1) == np.arange(len(P))).mean())


_ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                  (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
_mu = _ref.mean(0)
_u, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)
usable = [w for w in WIDTHS if w <= len(_sv)]
print(f"concat {_ref.shape}; {len(usable)} widths\n")


def hub_at(d):
    B = _VT[:d].T / (_sv[:d] / np.sqrt(len(_ref)))
    return (_ref - _mu) @ B


def sweep(src, tgt_name):
    T = TARGETS[tgt_name]
    gal = l2n(T[te])
    out = []
    for d in usable:
        H = hub_at(d)
        to_hub = {k: ridge(SPACES[k][tr], H) for k in SOURCES}
        head = ridge(SPACES[src][tr] @ to_hub[src], T[tr])
        pcts = []
        for enc in SOURCES:
            if enc == src:
                continue
            zero = r1((SPACES[enc][te] @ to_hub[enc]) @ head, gal)
            nat = r1(SPACES[enc][te] @ ridge(SPACES[enc][tr], T[tr]), gal)
            pcts.append(zero / max(nat, 1e-9))
        out.append(float(np.mean(pcts)))
    return np.array(out)


MIN_DROP = 0.02   # below this a curve is flat and HAS no largest drop


def onset(curve):
    """Where is the largest single drop, and how sharp is it?

    Location is the argmin of the step vector - no free parameter, and it
    is the pre-registered claim. Sharpness is reported alongside as pure
    description and is never a gate; conflating the two is what made G11
    misreport two curves that visibly cliff.

    One guard, and it is deliberately not a cliff criterion: on a flat
    curve argmin still returns an index, so a curve where nothing happens
    would be assigned a spurious location. If the largest drop is smaller
    than MIN_DROP the curve is reported as having none. That distinguishes
    "something collapsed" from "nothing did" - it does not decide WHICH
    drop is the cliff, which is the question G11 got wrong by putting a
    threshold in the wrong place.
    """
    steps = np.diff(curve)
    i = int(np.argmin(steps))
    if steps[i] > -MIN_DROP:
        return None, float(steps[i]), 0.0
    others = np.abs(np.delete(steps, i))
    sharp = min(abs(steps[i]) / (others.max() + 1e-12), 999.0)
    return usable[i + 1], float(steps[i]), float(sharp)


print("=" * 118)
hdr = f"{'source':11s}{'dim':>6s}{'target':>7s}  " + "".join(f"{w:>7d}" for w in usable)
print(hdr + f"{'drop at':>9s}{'size':>8s}{'sharp':>7s}")
print("=" * 118)
res = {}
for tgt in TARGETS:
    for src in SOURCES:
        c = sweep(src, tgt)
        at, size, sharp = onset(c)
        res[(src, tgt)] = (at, size, sharp, c)
        print(f"{src:11s}{DIM[src]:>6d}{tgt:>7s}  "
              + "".join(f"{v:>7.3f}" for v in c)
              + (f"{at:>9d}" if at else f"{'none':>9s}")
              + f"{size:>8.3f}{sharp:>7.1f}")

In [ ]:
# ---------- score ----------
print("\n" + "=" * 118)
hit = [k for k, v in res.items() if v[0] == DIM[k[0]]]
miss = [k for k, v in res.items() if v[0] != DIM[k[0]]]
flat = [k for k, v in res.items() if v[0] is None]
for src in SOURCES:
    locs = {res[(src, t)][0] for t in TARGETS}
    print(f"  {src:12s} dim {DIM[src]:5d}   largest drop at {locs}"
          + ("   HIT" if locs == {DIM[src]} else "   MISS"))

# do the new widths behave, i.e. is anything happening where nothing should?
print("\n  the three new widths, which the account says are unremarkable:")
bad_new = []
for w in sorted(NEW_POINTS):
    if w not in usable:
        continue
    j = usable.index(w)
    for (src, tgt), (_, _, _, c) in res.items():
        if j == 0:
            continue
        step = c[j] - c[j - 1]
        if step < -0.15 and w != DIM[src]:
            bad_new.append((w, src, tgt, step))
    print(f"    {w}: " + "  ".join(
        f"{s[:9]}/{t[:4]} {res[(s, t)][3][usable.index(w)]:.3f}"
        for s in SOURCES for t in TARGETS))
if bad_new:
    print("\n  UNEXPECTED COLLAPSE at a width that is nobody's dimension:")
    for w, s, t, st in bad_new:
        print(f"    {w}  {s}/{t}  step {st:+.3f}")

print()
if flat:
    print(f"\n  NO COLLAPSE AT ALL for: "
          + ", ".join(f"{s}/{t}" for s, t in flat))
    print("  The account predicts a collapse at that source's dimension, so")
    print("  a flat curve across the swept range contradicts it - unless the")
    print("  dimension lies outside the grid, in which case the run is")
    print("  uninformative for that source rather than contrary.")

if not miss and not bad_new:
    print("CONFIRMED, AND ON A GRID THAT COULD HAVE REFUTED IT.")
    print("Every source's largest drop sits at its own ambient dimension,")
    print("under both head targets, and the three added widths - which are")
    print("nobody's dimension - show nothing. The 'nowhere earlier' half of")
    print("the prediction now rests on samples that could have contradicted")
    print("it rather than on a gap in the grid.")
    print()
    print("The width cliff is a RANK WALL. A d-dimensional source's entry")
    print("map spans at most d hub directions; the head fitted on those")
    print("coordinates cannot read the rest, so past width d it is asked")
    print("for directions its source could never populate. Eighth account")
    print("tested, first to survive, and it survived out of sample.")
elif miss:
    print("NOT CONFIRMED: " + ", ".join(f"{s}/{t}" for s, t in miss))
    print("Report the table as measured. Do not keep the curves that fit.")
else:
    print("MOSTLY CONFIRMED, but something happens at a width that is")
    print("nobody's dimension. That needs explaining before the account is")
    print("used - an unexplained second collapse is a new open question.")